In [3]:
# ================================
# exp23_spectral_stacking
# ElasticNet + PLS + PCA Ridge stacking
# SIGNATE style experiment
# ================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error


# ================================
# Metric diagnostics tool
# ================================

def metric_diagnostics(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    y_true_log = np.log1p(y_true)
    y_pred_log = np.log1p(np.maximum(y_pred,0))
    rmsle = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    nrmse_mean = rmse / np.mean(y_true)
    nrmse_range = rmse / (np.max(y_true) - np.min(y_true))

    print("\nMetric diagnostics")
    print("------------------")

    print("RMSE:", rmse)
    print("MAE:", mae)
    print("RMSLE:", rmsle)
    print("NRMSE (mean):", nrmse_mean)
    print("NRMSE (range):", nrmse_range)


# ================================
# Load data (robust)
# ================================

def load_data():

    possible_train = [
        "../data/train.csv",
        "data/train.csv",
        "train.csv"
    ]

    possible_test = [
        "../data/test.csv",
        "data/test.csv",
        "test.csv"
    ]

    train_path = None
    test_path = None

    for p in possible_train:
        if os.path.exists(p):
            train_path = p
            break

    for p in possible_test:
        if os.path.exists(p):
            test_path = p
            break

    if train_path is None or test_path is None:
        raise FileNotFoundError("train/test file not found")

    print("Train:", train_path)
    print("Test:", test_path)

    train = pd.read_csv(train_path, encoding="cp932")
    test = pd.read_csv(test_path, encoding="cp932")

    return train, test


train, test = load_data()

target = "含水率"
id_col = "sample number"


# ================================
# Feature selection
# ================================

spectral_cols = [
    c for c in train.columns
    if c not in ["sample number", "species number", "樹種", "含水率"]
]

X = train[spectral_cols]
y = train[target]

X_test = test[spectral_cols]


# ================================
# Level 1 models
# ================================

elastic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=30000))
])

pls_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", PLSRegression(n_components=12))
])

pca_ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=50)),
    ("model", Ridge(alpha=10))
])


# ================================
# KFold setup
# ================================

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_elastic = np.zeros(len(X))
oof_pls = np.zeros(len(X))
oof_pca = np.zeros(len(X))

test_elastic = np.zeros(len(X_test))
test_pls = np.zeros(len(X_test))
test_pca = np.zeros(len(X_test))


# ================================
# Train Level 1 models
# ================================

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

    print(f"\nFold {fold+1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # ElasticNet
    elastic_model.fit(X_train, y_train)
    pred_val = elastic_model.predict(X_val)

    oof_elastic[val_idx] = pred_val
    test_elastic += elastic_model.predict(X_test) / kf.n_splits

    metric_diagnostics(y_val, pred_val)


    # PLS
    pls_model.fit(X_train, y_train)
    pred_val = pls_model.predict(X_val).ravel()

    oof_pls[val_idx] = pred_val
    test_pls += pls_model.predict(X_test).ravel() / kf.n_splits

    metric_diagnostics(y_val, pred_val)


    # PCA Ridge
    pca_ridge_model.fit(X_train, y_train)
    pred_val = pca_ridge_model.predict(X_val)

    oof_pca[val_idx] = pred_val
    test_pca += pca_ridge_model.predict(X_test) / kf.n_splits

    metric_diagnostics(y_val, pred_val)


# ================================
# Level 2 stacking dataset
# ================================

stack_train = pd.DataFrame({
    "elastic": oof_elastic,
    "pls": oof_pls,
    "pca_ridge": oof_pca
})

stack_test = pd.DataFrame({
    "elastic": test_elastic,
    "pls": test_pls,
    "pca_ridge": test_pca
})


# ================================
# Meta model
# ================================

meta_model = Ridge(alpha=1)

meta_model.fit(stack_train, y)

final_pred = meta_model.predict(stack_train)

print("\nStacking performance:")
metric_diagnostics(y, final_pred)


# ================================
# Test predictions
# ================================

test_predictions = meta_model.predict(stack_test)


# ================================
# Save submission
# ================================

os.makedirs("../submissions", exist_ok=True)

submission = pd.DataFrame({
    "sample number": test[id_col],
    "含水率": test_predictions
})

output_path = "../submissions/exp23_spectral_stacking_20260326.csv"

submission.to_csv(output_path, index=False, header=False)

print("\nSubmission saved:")
print(output_path)
print(submission.head())

Train: ../data/train.csv
Test: ../data/test.csv

Fold 1


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.576e+05, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(



Metric diagnostics
------------------
RMSE: 14.15154544588829
MAE: 9.445474793441612
RMSLE: 0.5850454506900961
NRMSE (mean): 0.2835399920430497
NRMSE (range): 0.050615247761711335

Metric diagnostics
------------------
RMSE: 15.679606650145926
MAE: 10.892603626513898
RMSLE: 0.6074433193162921
NRMSE (mean): 0.31415618610843976
NRMSE (range): 0.0560806011214768

Metric diagnostics
------------------
RMSE: 15.813254088430817
MAE: 10.99872075526117
RMSLE: 0.646449633095904
NRMSE (mean): 0.31683394266391773
NRMSE (range): 0.056558612390802385

Fold 2


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.504e+05, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(



Metric diagnostics
------------------
RMSE: 15.720732911414864
MAE: 9.657524235885003
RMSLE: 0.6996207085761537
NRMSE (mean): 0.29918948458585565
NRMSE (range): 0.05470136297977716

Metric diagnostics
------------------
RMSE: 17.074500865113517
MAE: 11.591938916298675
RMSLE: 0.6551250783285782
NRMSE (mean): 0.324953750068787
NRMSE (range): 0.05941189095852618

Metric diagnostics
------------------
RMSE: 17.229122132465854
MAE: 11.534683946684545
RMSLE: 0.7212905921632132
NRMSE (mean): 0.3278964282216336
NRMSE (range): 0.05994990620994565

Fold 3


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.457e+05, tolerance: 2.603e+02
  model = cd_fast.enet_coordinate_descent(



Metric diagnostics
------------------
RMSE: 15.504517340171976
MAE: 9.995125267111527
RMSLE: 0.5688485576992945
NRMSE (mean): 0.32372015190404774
NRMSE (range): 0.06552558064080338

Metric diagnostics
------------------
RMSE: 18.031330908297953
MAE: 12.431527823384789
RMSLE: 0.6572446828549591
NRMSE (mean): 0.3764777098570177
NRMSE (range): 0.0762044636134144

Metric diagnostics
------------------
RMSE: 18.977042029356042
MAE: 13.076063061298882
RMSLE: 0.7469430260853137
NRMSE (mean): 0.39622329374391835
NRMSE (range): 0.0802012517085354

Fold 4


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.611e+05, tolerance: 2.636e+02
  model = cd_fast.enet_coordinate_descent(



Metric diagnostics
------------------
RMSE: 12.993177249022532
MAE: 8.373947267924072
RMSLE: 0.6166924689678921
NRMSE (mean): 0.27799358124813506
NRMSE (range): 0.043824695415608204

Metric diagnostics
------------------
RMSE: 15.528780984645305
MAE: 10.493618817722876
RMSLE: 0.7246590151292104
NRMSE (mean): 0.3322437118807293
NRMSE (range): 0.052377034791776265

Metric diagnostics
------------------
RMSE: 15.731066504889112
MAE: 10.694962678122568
RMSLE: 0.7313851534639426
NRMSE (mean): 0.3365716814858118
NRMSE (range): 0.05305932374556851

Fold 5

Metric diagnostics
------------------
RMSE: 15.6622403545643
MAE: 9.883039790629491
RMSLE: 0.6334007801612274
NRMSE (mean): 0.29781775142495415
NRMSE (range): 0.07461323833692182

Metric diagnostics
------------------
RMSE: 18.55346321920651
MAE: 11.912852358775963
RMSLE: 0.6019117590007678
NRMSE (mean): 0.35279440054560357
NRMSE (range): 0.0883867149150565

Metric diagnostics
------------------
RMSE: 18.843707488273214
MAE: 11.93167303665

/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.477e+05, tolerance: 2.606e+02
  model = cd_fast.enet_coordinate_descent(
